# PhoBERT 4-class SMS Shield VN

Google Colab GPU fine-tuning notebook. Labels: `0=LEGITIMATE`, `1=PROMOTION`, `2=PHISHING`, `3=FINANCIAL_SCAM`.

## 0. Scope and safety

This notebook uses only the Phase 1 locked dataset. It does not download or merge new SMS data, alter labels, use test data for model selection, or replace the production binary models. It stops before training if the lock manifest, split hashes, four labels, or provenance checks are invalid.

## 1. Environment setup

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, time
import pandas as pd

PROJECT_ROOT = Path("/content/SMS_Scam_Detector_NLP")
CONFIG_PATH = PROJECT_ROOT / "notebooks" / "colab" / "config.json"
REQUIREMENTS_PATH = PROJECT_ROOT / "notebooks" / "colab" / "requirements-colab.txt"
if not PROJECT_ROOT.exists():
    raise FileNotFoundError("Upload or mount the repository, then set PROJECT_ROOT to its path.")
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
print("project:", PROJECT_ROOT)
print("python:", sys.version)
print("config model:", config["model_name"])

In [ ]:
# Colab supplies PyTorch/CUDA. Install the project-specific packages only.
!pip -q install -r "{REQUIREMENTS_PATH}"

## 2. GPU verification

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GiB:", round(props.total_memory / 1024**3, 2))
    device = torch.device("cuda")
else:
    print("GPU is unavailable. This cell permits a CPU smoke test only; it does not claim GPU results.")
    device = torch.device("cpu")
subprocess.run(["nvidia-smi"], check=False)

## 3. Google Drive mounting

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_EXPERIMENT_ROOT = Path("/content/drive/MyDrive/SMS_Shield_VN/experiments/phobert")
except ImportError:
    DRIVE_EXPERIMENT_ROOT = PROJECT_ROOT / "experiments" / "phobert"
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = DRIVE_EXPERIMENT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("checkpoint directory:", RUN_DIR)

## 4. Dataset loading

In [ ]:
LABEL2ID = config["label2id"]
ID2LABEL = {int(value): key for key, value in LABEL2ID.items()}
SPLITS = ["train", "validation", "test"]
split_frames = {}
for split in SPLITS:
    path = PROJECT_ROOT / "data" / "processed" / f"{split}.csv"
    split_frames[split] = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    print(split, len(split_frames[split]), split_frames[split]["label"].value_counts().to_dict())

## 5. Dataset validation

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

lock_path = PROJECT_ROOT / "data" / "manifests" / "dataset_lock.json"
if not lock_path.exists():
    raise RuntimeError("Missing dataset_lock.json. Complete Phase 1 annotation and lock first.")
lock = json.loads(lock_path.read_text(encoding="utf-8"))
for split in SPLITS:
    path = PROJECT_ROOT / "data" / "processed" / f"{split}.csv"
    expected = lock["files"][split]["sha256"]
    actual = sha256_file(path)
    if expected != actual:
        raise RuntimeError(f"Locked hash mismatch for {split}: {actual} != {expected}")
    frame = split_frames[split]
    if frame.empty or set(frame["label"]) != set(LABEL2ID):
        raise RuntimeError(f"{split} must be non-empty and contain all four labels.")
    if frame["source_type"].eq("synthetic").any() and split != "train":
        raise RuntimeError("Synthetic records are allowed only in train.")
    if not frame["label"].map(LABEL2ID).notna().all():
        raise RuntimeError(f"Invalid label in {split}.")
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    left_hashes = set(split_frames[left]["message"].map(lambda x: hashlib.sha256(str(x).strip().casefold().encode("utf-8")).hexdigest()))
    right_hashes = set(split_frames[right]["message"].map(lambda x: hashlib.sha256(str(x).strip().casefold().encode("utf-8")).hexdigest()))
    if left_hashes & right_hashes:
        raise RuntimeError(f"Normalized text leakage: {left} vs {right}")
    if set(split_frames[left]["group_id"]) & set(split_frames[right]["group_id"]):
        raise RuntimeError(f"Group leakage: {left} vs {right}")
print("Dataset lock, labels, source policy, hashes and split leakage checks passed.")

## 6. Vietnamese preprocessing and 7. Word segmentation

In [ ]:
# VnCoreNLP RDRSegmenter is used consistently for every split.
import py_vncorenlp
VNCORERLP_DIR = Path("/content/vncorenlp")
if not (VNCORERLP_DIR / "VnCoreNLP-1.2.jar").exists():
    py_vncorenlp.download_model(save_dir=str(VNCORERLP_DIR))
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORERLP_DIR))
print(segmenter.word_segment("Chúng tôi đang nghiên cứu bảo mật."))

cache_path = PROJECT_ROOT / config["segmentation"]["cache_file"]
cache_path.parent.mkdir(parents=True, exist_ok=True)
seg_cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}

def segment_text(text):
    normalized = " ".join(str(text).replace("\u200b", " ").split())
    key = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    if key not in seg_cache:
        seg_cache[key] = segmenter.word_segment(normalized)
    return seg_cache[key]

for split, frame in split_frames.items():
    frame["segmented_message"] = frame["message"].map(segment_text)
cache_path.write_text(json.dumps(seg_cache, ensure_ascii=False, indent=2), encoding="utf-8")
print("segmentation cache entries:", len(seg_cache))

## 8. Tokenization

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = config["model_name"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenized = {}
for split, frame in split_frames.items():
    data = frame[["segmented_message", "label"]].copy()
    data["labels"] = data["label"].map(LABEL2ID).astype(int)
    dataset = Dataset.from_pandas(data[["segmented_message", "labels"]], preserve_index=False)
    dataset = dataset.map(lambda batch: tokenizer(batch["segmented_message"], truncation=True, max_length=config["max_length"]), batched=True)
    tokenized[split] = dataset
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized)

## 9. Model loading

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=config["num_labels"],
    id2label={str(k): v for k, v in ID2LABEL.items()},
    label2id=LABEL2ID,
)
model.config.id2label = {int(k): v for k, v in model.config.id2label.items()}
model.to(device)
class_counts = split_frames["train"]["label"].map(LABEL2ID).value_counts().reindex(range(config["num_labels"]), fill_value=0)
weights = len(split_frames["train"]) / (config["num_labels"] * class_counts.replace(0, 1))
class_weights = torch.tensor(weights.to_numpy(), dtype=torch.float, device=device)
print(model.config.id2label)
print("class weights:", class_weights.detach().cpu().tolist())

## 10. Training configuration

In [ ]:
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.preprocessing import label_binarize
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed

SEED = int(config["seed"])
set_seed(SEED)
bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
fp16 = bool(torch.cuda.is_available() and not bf16)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, labels=list(ID2LABEL), zero_division=0)
    high_true, high_pred = np.isin(labels, [2, 3]), np.isin(predictions, [2, 3])
    tn, fp, fn, tp = __import__("sklearn").metrics.confusion_matrix(high_true, high_pred, labels=[False, True]).ravel()
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "high_risk_precision": float(__import__("sklearn").metrics.precision_score(high_true, high_pred, zero_division=0)),
        "high_risk_recall": float(__import__("sklearn").metrics.recall_score(high_true, high_pred, zero_division=0)),
        "high_risk_fpr": float(fp / (fp + tn)) if fp + tn else 0.0,
        "high_risk_fnr": float(fn / (fn + tp)) if fn + tp else 0.0,
        "multiclass_pr_auc_ovr": float(average_precision_score(label_binarize(labels, classes=list(ID2LABEL)), probabilities, average="macro")),
        **{f"precision_{ID2LABEL[i]}": float(precision[i]) for i in range(4)},
        **{f"recall_{ID2LABEL[i]}": float(recall[i]) for i in range(4)},
        **{f"f1_{ID2LABEL[i]}": float(f1[i]) for i in range(4)},
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_kwargs = dict(
    output_dir=str(RUN_DIR),
    num_train_epochs=config["epochs"],
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["train_batch_size"] if torch.cuda.is_available() else 2,
    per_device_eval_batch_size=config["eval_batch_size"] if torch.cuda.is_available() else 2,
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    weight_decay=config["weight_decay"],
    warmup_ratio=config["warmup_ratio"],
    max_grad_norm=1.0,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=fp16,
    bf16=bf16,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)
try:
    training_args = TrainingArguments(**training_kwargs)
except TypeError:
    training_kwargs["eval_strategy"] = training_kwargs.pop("evaluation_strategy")
    training_args = TrainingArguments(**training_kwargs)

## 10b. Training

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])],
)
train_started = time.time()
train_result = trainer.train()
train_seconds = time.time() - train_started
history = pd.DataFrame(trainer.state.log_history)
history.to_csv(RUN_DIR / "training_log.csv", index=False)
print("training seconds:", round(train_seconds, 2))
print("best checkpoint:", trainer.state.best_model_checkpoint)

## 11. Validation

In [ ]:
validation_metrics = trainer.evaluate(tokenized["validation"], metric_key_prefix="validation")
print(json.dumps(validation_metrics, indent=2, ensure_ascii=False))
pd.DataFrame([validation_metrics]).to_csv(RUN_DIR / "validation_metrics.csv", index=False)

## 12. Test evaluation

In [ ]:
# Test is evaluated only after the checkpoint was selected by validation Macro-F1.
test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
pd.DataFrame([test_metrics]).to_csv(RUN_DIR / "test_metrics.csv", index=False)

## 13. Error analysis

In [ ]:
prediction_output = trainer.predict(tokenized["test"])
test_predictions = np.argmax(prediction_output.predictions, axis=-1)
test_truth = split_frames["test"]["label"].map(LABEL2ID).to_numpy()
errors = split_frames["test"].copy()
errors["true_label"] = test_truth
errors["predicted_label"] = test_predictions
errors["true_label_name"] = errors["true_label"].map(ID2LABEL)
errors["predicted_label_name"] = errors["predicted_label"].map(ID2LABEL)
errors = errors[errors["true_label"] != errors["predicted_label"]]
errors.to_csv(RUN_DIR / "false_predictions.csv", index=False, encoding="utf-8-sig")
confusion = errors.groupby(["true_label_name", "predicted_label_name"]).size().reset_index(name="count")
confusion.to_csv(RUN_DIR / "class_confusion_analysis.csv", index=False)
print("errors:", len(errors))
display(errors[["sample_id", "message", "true_label_name", "predicted_label_name"]].head(30))

## 14. Model export

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "models" / "phobert_4class"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
(EXPORT_DIR / "label_map.json").write_text(json.dumps({"label2id": LABEL2ID, "id2label": ID2LABEL}, ensure_ascii=False, indent=2), encoding="utf-8")
(EXPORT_DIR / "preprocessing_config.json").write_text(json.dumps({"unicode": "NFC", "word_segmentation": config["segmentation"], "max_length": config["max_length"]}, ensure_ascii=False, indent=2), encoding="utf-8")
metrics_bundle = {"validation": validation_metrics, "test": test_metrics, "train_seconds": train_seconds, "best_checkpoint": trainer.state.best_model_checkpoint}
(EXPORT_DIR / "metrics.json").write_text(json.dumps(metrics_bundle, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
manifest = {
    "model_name": MODEL_NAME,
    "model_version": f"phobert_base_{RUN_ID}",
    "run_dir": str(RUN_DIR),
    "label_mapping": LABEL2ID,
    "dataset_version": sha256_file(lock_path),
    "dataset_lock_sha256": sha256_file(lock_path),
    "dataset_files": {split: sha256_file(PROJECT_ROOT / "data" / "processed" / f"{split}.csv") for split in SPLITS},
    "preprocessing_version": "phase3_colab_v1",
    "training_config": config,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "calibration": {"status": "pending_phase4"},
    "threshold": {"status": "pending_phase4"},
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
(EXPORT_DIR / "model_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("exported:", EXPORT_DIR)

## 15. CPU smoke test

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
cpu_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR, local_files_only=True)
cpu_model = AutoModelForSequenceClassification.from_pretrained(EXPORT_DIR, local_files_only=True).to("cpu")
sample = "Ngân hàng không yêu cầu cung cấp OTP cho bất kỳ ai."
inputs = cpu_tokenizer(sample, return_tensors="pt", truncation=True, max_length=config["max_length"])
with torch.no_grad():
    prediction_id = int(cpu_model(**inputs).logits.argmax(dim=-1).item())
print({"cpu_prediction_id": prediction_id, "cpu_prediction_label": ID2LABEL[prediction_id]})

## 16. Comparison with Phase 2

In [ ]:
benchmark_path = PROJECT_ROOT / "results" / "benchmark" / "phobert_vs_classical_ml.csv"
benchmark_path.parent.mkdir(parents=True, exist_ok=True)
classical_path = PROJECT_ROOT / "results" / "classical_ml" / "test_metrics.csv"
rows = [{"model": "PhoBERT", "macro_f1": test_metrics.get("test_macro_f1"), "weighted_f1": test_metrics.get("test_weighted_f1"), "accuracy": test_metrics.get("test_accuracy"), "high_risk_recall": test_metrics.get("test_high_risk_recall")}]
if classical_path.exists():
    classical = pd.read_csv(classical_path)
    if not classical.empty and "model_id" in classical.columns:
        for _, row in classical.iterrows():
            rows.append({"model": row.get("model_id"), "macro_f1": row.get("macro_f1"), "weighted_f1": row.get("weighted_f1"), "accuracy": row.get("accuracy"), "high_risk_recall": row.get("high_risk_recall")})
pd.DataFrame(rows).to_csv(benchmark_path, index=False)
print("comparison written:", benchmark_path)